In [ ]:
import meteostat as ms
from meteostat import stations, Point
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd
import numpy as np

output_file = #path to where you save merged file


#change year to get different year of data
#range of dates to pull data from 
start = datetime(2025, 1, 1)
end = datetime(2025, 12, 31, 23, 59)

airports = {
    "TPA": (27.9755, -82.5332),
    "STL": (38.7487, -90.3700),
    "SNA": (33.6757, -117.8678),
    "SMF": (38.6954, -121.5908),
    "SLC": (40.7899, -111.9791),
    "SJC": (37.3639, -121.9289),
    "SFO": (37.6213, -122.3790),
    "SEA": (47.4502, -122.3088),
    "SAT": (29.5337, -98.4698),
    "SAN": (32.7338, -117.1933),
    "RSW": (26.5362, -81.7552),
    "RDU": (35.8801, -78.7880),
    "PIT": (40.4915, -80.2329),
    "PHX": (33.4342, -112.0116),
    "PHL": (39.8744, -75.2424),
    "PDX": (45.5898, -122.5951),
    "ORD": (41.9742, -87.9073),
    "OGG": (20.8987, -156.4305),
    "OAK": (37.7213, -122.2210),
    "MSY": (29.9934, -90.2580),
    "MSP": (44.8848, -93.2223),
    "MIA": (25.7959, -80.2870),
    "MDW": (41.7868, -87.7522),
    "MCO": (28.4312, -81.3081),
    "MCI": (39.2976, -94.7139),
    "LGA": (40.7769, -73.8740),
    "LAX": (33.9416, -118.4085),
    "LAS": (36.0840, -115.1537),
    "JFK": (40.6413, -73.7781),
    "IND": (39.7173, -86.2944),
    "IAH": (29.9902, -95.3368),
    "IAD": (38.9531, -77.4565),
    "HOU": (29.6454, -95.2789),
    "HNL": (21.3187, -157.9225),
    "FLL": (26.0726, -80.1527),
    "EWR": (40.6895, -74.1745),
    "DTW": (42.2162, -83.3554),
    "DFW": (32.8998, -97.0403),
    "DEN": (39.8561, -104.6737),
    "DCA": (38.8512, -77.0402),
    "DAL": (32.8471, -96.8517),
    "CVG": (39.0488, -84.6678),
    "CMH": (39.9980, -82.8919),
    "CLT": (35.2144, -80.9473),
    "CLE": (41.4117, -81.8498),
    "BWI": (39.1754, -76.6684),
    "BOS": (42.3656, -71.0096),
    "BNA": (36.1263, -86.6774),
    "AUS": (30.1975, -97.6664),
    "ATL": (33.6407, -84.4277)
}


#since it's a large amount of data, split into
# 30 day chunks to process data
def chunk_period(start, end, days=30):
    chunks = []
    cur = start

    while cur < end:
        nxt = min(cur + timedelta(days=days), end)
        chunks.append((cur, nxt))
        cur = nxt

    return chunks


all_data = []

#loop through airports
for code, (lat, lon) in airports.items():
    print(f"Processing {code}...")

    location = Point(lat, lon)

    #finds weather station nearest airport
    nearest = stations.nearby(location).head(1)

    if nearest.empty:
        print(f"⚠️ No station for {code}")
        continue

    station_id = nearest.index[0]

    frames = []

    #loop through chunks to get weather data
    for s, e in chunk_period(start, end, days=30):
        df = ms.hourly(station_id, s, e).fetch()

        if df is not None and not df.empty:
            frames.append(df)

    if not frames:
        print(f"⚠️ No data for {code}")
        continue

    df = pd.concat(frames).reset_index()
    df["airport"] = code

    all_data.append(df)


#clean and format output
if all_data:
    final = pd.concat(all_data, ignore_index=True)

    # split datetime index into date and time
    final["date"] = final["time"].dt.date
    final["time"] = final["time"].dt.time

    # rename columns 
    rename_map = {
        "temp": "temperature",
        "rhum": "humidity",
        "prcp": "precipitation",
        "wdir": "wind_direction",
        "wspd": "wind_speed",
        "pres": "pressure",
        "cldc": "cloud_cover",
        "coco": "weather_condition_code"
    }

    final = final.rename(columns=rename_map)

    #ensure airport column is uppercase
    final["airport"] = final["airport"].str.upper()

    keep_cols = [
    "airport", "date", "time",
    "temperature", "humidity",
    "precipitation", "wind_direction", "wind_speed",
    "pressure", "cloud_cover", "weather_condition_code"
    ]

    #make sure we only keep columns we want
    final = final.reindex(columns=keep_cols)

    #  reorder columns nicely
    cols = ["airport", "date", "time"] + [c for c in final.columns if c not in ["airport", "date", "time"]]
    final = final[cols]

    
    final = final.replace({pd.NA: np.nan})
    final.to_csv(output_file, index=False, na_rep="NaN")

    print(f"Saved to path")
else:
    print("No data collected.")



In [ ]:
#there were 600 duplicates in 2023 data, this is handling to get rid of them
import pandas as pd
from pathlib import Path

DATA_DIR = Path.home() 
INPUT = DATA_DIR / "2023_hourly_weather_data.csv"
OUTPUT = DATA_DIR / "2023_hourly_weather_data_CLEAN.csv"


df = pd.read_csv(INPUT)

#commented out section below inspected the duplicate rows
# dup_all = df[df.duplicated(subset=["airport", "date", "time"], keep=False)] \
#             .sort_values(["airport", "date", "time"])

# dup_all

dup_rows = df[df.duplicated(subset=["airport", "date", "time"], keep=False)]

print("Total duplicate rows:", len(dup_rows))

removed_examples = dup_rows.sort_values(["airport", "date", "time"])

removed_examples.head(30)

dup_rows.sort_values(["airport", "date", "time"]).head(20)

df[df.duplicated(subset=["airport", "date", "time"], keep=False)] \
  .groupby("airport").size().sort_values(ascending=False)

#use drop function to get rid of duplicates. This keeps the 
#first instance that the duplicates occur. 
df_clean = df.drop_duplicates(subset=["airport", "date", "time"])

print("Remaining duplicates:",
      df_clean.duplicated(subset=["airport", "date", "time"]).sum())

df_clean.to_csv(OUTPUT, index=False)




Total duplicate rows: 1200
Remaining duplicates: 0


In [ ]:
#handling for 2025 data 
#there were 134 missing rows here. 
#I added them in, hourly, with NaN values for everything
#except airport, hour, and date columns. 

import pandas as pd
from pathlib import Path

DATA_DIR = Path.home() / "OneDrive" / "Desktop" / "pythonfiles" / "engg2112project" / "finished_datasets_to_use"

INPUT = DATA_DIR / "2025_hourly_weather_data_CLEAN2.csv"
OUTPUT = DATA_DIR / "2025_hourly_weather_data_correct_padded.csv"

df = pd.read_csv(INPUT)

# get number of rows before
num_rows = len(df)
print("Number of rows:", num_rows)


#clean hour, date, and time columns
# Date
df['date'] = pd.to_datetime(df['date'])

# hour
df['hour'] = (
    df['hour']
    .astype(str)
    .str.extract(r'(\d{1,2})')[0]   # pulls "00" from "00:00"
)
df['hour'] = pd.to_numeric(df['hour'], errors='coerce')

# Drop bad rows
df = df.dropna(subset=['airport', 'date', 'hour'])

# Convert to int
df['hour'] = df['hour'].astype(int)

# Keep only valid hours
df = df[df['hour'].between(0, 23)]

print("After cleaning rows:", len(df))

#remove duplicates
# ensure one row per airport/date/hour
df = df.groupby(['airport', 'date', 'hour'], as_index=False).mean(numeric_only=True)
print("After deduplication rows:", len(df))

#validate types
print("Hour dtype:", df['hour'].dtype)
print("Hour sample:", df['hour'].unique()[:10])
assert df['hour'].between(0, 23).all(), "Invalid hour values exist"



#create full grid with all unique hour/airport/date combinations
airports = df['airport'].unique()

full_dates = pd.date_range(
    start=df['date'].min(),
    end=df['date'].max(),
    freq='D'
)

hours = list(range(24))

full_index = pd.MultiIndex.from_product(
    [airports, full_dates, hours],
    names=['airport', 'date', 'hour']
)

full_df = pd.DataFrame(index=full_index).reset_index()

print("Expected full rows:", len(full_df))


#pad missing values
df_full = df.set_index(['airport', 'date', 'hour'])
full_df_indexed = full_df.set_index(['airport', 'date', 'hour'])

df_completed = df_full.reindex(full_df_indexed.index).reset_index()

#ensure all rows are there
print("Final rows:", len(df_completed))
print("Rows added:", len(df_completed) - len(df))


#ensure columns are what the merge code expects to recieve
df_completed["time"] = df_completed["hour"].apply(lambda h: f"{int(h):02d}:00")
df_completed = df_completed.drop(columns=["hour"])

print(df_completed.head())

#write to csv and print values
print("Saved")

original_rows = len(df)
new_rows = len(df_completed)

print("Rows added:", new_rows - original_rows)

df_completed.to_csv(OUTPUT, index=False)

Number of rows: 437866
After cleaning rows: 437866
After deduplication rows: 437866
Hour dtype: int64
Hour sample: [0 1 2 3 4 5 6 7 8 9]
Expected full rows: 438000
Final rows: 438000
Rows added: 134
  airport       date  temperature  humidity  precipitation  wind_direction  \
0     ATL 2025-01-01         12.0      50.0            0.0           290.0   
1     ATL 2025-01-01         11.7      50.0            0.0           290.0   
2     ATL 2025-01-01         10.6      52.0            0.0           290.0   
3     ATL 2025-01-01         10.0      54.0            0.0           300.0   
4     ATL 2025-01-01          9.4      52.0            0.0           310.0   

   wind_speed  pressure  cloud_cover  weather_condition_code   time  
0        30.0    1013.0          0.0                     1.0  00:00  
1        29.5    1012.4          0.0                     1.0  01:00  
2        20.5    1013.3          0.0                     1.0  02:00  
3        24.1    1013.5          0.0                